In [1]:
import os
import sys

In [2]:
sys.path.append(os.path.abspath("../.."))

In [3]:
from datalemur_utils import extract_iframe_from_url

In [4]:
url = "https://datalemur.com/questions/sql-average-post-hiatus-1"
extract_iframe_from_url(url)

In [5]:
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/03 02:36:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
posts = spark.read\
.option("header", "true")\
.option("inferSchema", "false")\
.option("delimiter", "\t")\
.schema("user_id int, post_id int, post_content string, post_date string")\
.format("csv")\
.load("posts.txt")

In [8]:
posts.createOrReplaceTempView("posts")

In [30]:
spark.sql(
    """
    SELECT 
    user_id, 
    datediff(MAX(to_timestamp(post_date, 'MM/dd/yyyy HH:mm:ss')), MIN(to_timestamp(post_date, 'MM/dd/yyyy HH:mm:ss'))) AS days_between
    FROM posts
    GROUP BY user_id
    HAVING MAX(to_timestamp(post_date, 'MM/dd/yyyy HH:mm:ss')) <> MIN(to_timestamp(post_date, 'MM/dd/yyyy HH:mm:ss'))
    """
).show()

+-------+------------+
|user_id|days_between|
+-------+------------+
| 151652|           2|
| 661093|          21|
+-------+------------+

